# Health Risk Classification

## Name: Noman Ali
## Roll Number: 24F-AI-202 (A1)
## Subject: Machine learning

Perform data preprocessing (handle missing values, normalize numeric data).

In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

df = pd.read_csv("diabetes.csv")
print("Explicit missing values per column:\n", df.isnull().sum())
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("\nZeros (hidden missing values) per column:\n", (df[zero_as_missing] == 0).sum())
df[zero_as_missing] = df[zero_as_missing].replace(0, np.nan)

X = df.drop("Outcome", axis=1)
y = df["Outcome"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X.columns, index=X_test.index)
print("\nMissing values after imputation:", X_train.isnull().sum().sum() + X_test.isnull().sum().sum())
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Explicit missing values per column:
 Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

Zeros (hidden missing values) per column:
 Glucose          0
BloodPressure    0
SkinThickness    0
Insulin          0
BMI              0
dtype: int64

Missing values after imputation: 0


Apply SVM and Decision Tree for classification

In [20]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

dt_unpruned = DecisionTreeClassifier(random_state=42)
dt_unpruned.fit(X_train, y_train)

dt_pruned = DecisionTreeClassifier(
    criterion="gini", max_depth=4, min_samples_split=10, random_state=42
)
dt_pruned.fit(X_train, y_train)

dt_train_preds = dt_pruned.predict(X_train)
dt_test_preds = dt_pruned.predict(X_test)

svm_model = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svm_model.fit(X_train_scaled, y_train)

svm_train_preds = svm_model.predict(X_train_scaled)
svm_test_preds = svm_model.predict(X_test_scaled)

Compare both models’ results and discuss overfitting and accuracy trade-offs

In [21]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


print("             DECISION TREE EVALUATION")
print(f"Unpruned Train Accuracy : {accuracy_score(y_train, dt_unpruned.predict(X_train)):.4f}")
print(f"Unpruned Test Accuracy  : {accuracy_score(y_test, dt_unpruned.predict(X_test)):.4f}")
print(f"Pruned Train Accuracy   : {accuracy_score(y_train, dt_train_preds):.4f}")
print(f"Pruned Test Accuracy    : {accuracy_score(y_test, dt_test_preds):.4f}\n")
print("Pruned Decision Tree Classification Report:")
print(classification_report(y_test, dt_test_preds))
print("             SUPPORT VECTOR MACHINE (SVM)")
print(f"SVM Train Accuracy      : {accuracy_score(y_train, svm_train_preds):.4f}")
print(f"SVM Test Accuracy       : {accuracy_score(y_test, svm_test_preds):.4f}\n")
print("SVM Classification Report:")
print(classification_report(y_test, svm_test_preds))

print("SVM Confusion Matrix:\n", confusion_matrix(y_test, svm_test_preds))

             DECISION TREE EVALUATION
Unpruned Train Accuracy : 1.0000
Unpruned Test Accuracy  : 0.8117
Pruned Train Accuracy   : 0.9055
Pruned Test Accuracy    : 0.8377

Pruned Decision Tree Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.89      0.88       100
           1       0.78      0.74      0.76        54

    accuracy                           0.84       154
   macro avg       0.82      0.82      0.82       154
weighted avg       0.84      0.84      0.84       154

             SUPPORT VECTOR MACHINE (SVM)
SVM Train Accuracy      : 0.8958
SVM Test Accuracy       : 0.8377

SVM Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.87      0.87       100
           1       0.76      0.78      0.77        54

    accuracy                           0.84       154
   macro avg       0.82      0.82      0.82       154
weighted avg       0.84      0.84      0.84       

###1: Decision Tree Overfitting & Pruning

The unpruned tree memorizes training data (100% accuracy) but drops to 81.2% on the test set due to overfitting. Restricting depth (`max_depth=4`) prevents excess branching, improving test accuracy to 83.8%.

###2: SVM Margin Maximization & Scaling

SVM achieves a balanced 89.6% train and 83.8% test accuracy by maximizing the margin via support vectors instead of isolating every point. Feature standardization ensures large-scale variables do not bias the RBF kernel distance calculations.